# ETL Awal 

Fokus notebook ini adalah eksplorasi, pembersihan, preprocessing, dan ETL lokal awal sesuai rencana Week 1. Data harga bulanan diubah dari format lebar ke format panjang: satu baris untuk satu komoditas, wilayah, dan tanggal.

Hasil disimpan sebagai CSV terkompresi (`.csv.gz`) di `etl/data/processed`. Notebook ini belum mengirim data ke MongoDB atau MinIO dan belum menghitung analitik lanjutan.

## 1. Konfigurasi dan lokasi data

Jalankan sel dari atas ke bawah. Dataset terdiri atas CSV bulanan 2025 di `etl/data/dataset`. Paket yang dipakai: pandas dan numpy.

In [1]:
from pathlib import Path
import pandas as pd

MONTHS = ['januari', 'februari', 'maret', 'april', 'mei', 'juni', 'juli', 'agustus', 'september', 'oktober', 'november', 'desember']
REQUIRED_COLUMNS = ['No', 'Kode Provinsi', 'Provinsi', 'Kode Daerah', 'Kabupaten Kota', 'COICOP', 'Komoditas']
COLUMNS_RENAME = {
    'Kode Provinsi': 'kode_provinsi',
    'Provinsi': 'provinsi',
    'Kode Daerah': 'kode_daerah',
    'Kabupaten Kota': 'kabupaten_kota',
    'COICOP': 'kode_komoditas',
    'Komoditas': 'komoditas',
}

data_candidates = [Path.cwd() / 'etl' / 'data' / 'dataset', Path.cwd() / 'data' / 'dataset']
DATA_DIR = next((path.resolve() for path in data_candidates if path.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Folder etl/data/dataset tidak ditemukan. Jalankan notebook dari root proyek atau folder etl.')

OUTPUT_DIR = DATA_DIR.parent / 'processed'
INPUT_FILES = [DATA_DIR / f'{month}.csv' for month in MONTHS]
print(f'Folder input : {DATA_DIR}')
print(f'Folder output: {OUTPUT_DIR}')

Folder input : C:\TUGAS\Semester5\BANJUT\PanganPulse\etl\data\dataset
Folder output: C:\TUGAS\Semester5\BANJUT\PanganPulse\etl\data\processed


## 2. Eksplorasi dan validasi sumber

Periksa kelengkapan file, nama kolom, jumlah baris, serta rentang tanggal pada tiap bulan sebelum melakukan transformasi.

In [ ]:
def inspect_dataset():
    missing_files = [path.name for path in INPUT_FILES if not path.is_file()]
    if missing_files:
        raise FileNotFoundError(f'File dataset belum lengkap: {missing_files}')

    rows = []
    for path in INPUT_FILES:
        columns = pd.read_csv(path, nrows=0).columns.tolist()
        missing_columns = [column for column in REQUIRED_COLUMNS if column not in columns]
        if missing_columns:
            raise ValueError(f'{path.name}: kolom wajib tidak ada: {missing_columns}')

        date_columns = [column for column in columns if column not in REQUIRED_COLUMNS]
        dates = pd.to_datetime(date_columns, format='%d/%m/%y', errors='coerce')
        if not date_columns or dates.isna().any():
            raise ValueError(f'{path.name}: ada kolom tanggal yang tidak valid.')

        rows.append({
            'file': path.name,
            'jumlah_baris': sum(1 for _ in path.open(encoding='utf-8-sig')) - 1,
            'jumlah_tanggal': len(date_columns),
            'tanggal_awal': dates.min().date(),
            'tanggal_akhir': dates.max().date(),
        })
    return pd.DataFrame(rows)


dataset_overview = inspect_dataset()
display(dataset_overview)
print(f"Total baris sumber: {dataset_overview['jumlah_baris'].sum():,}")

,file,jumlah_baris,jumlah_tanggal,tanggal_awal,tanggal_akhir
0,januari.csv,12479,19,2025-01-02,2025-01-31
1,februari.csv,12518,20,2025-02-03,2025-02-28
2,maret.csv,12521,19,2025-03-03,2025-03-27
3,april.csv,12473,16,2025-04-08,2025-04-30
4,mei.csv,12473,17,2025-05-02,2025-05-28
5,juni.csv,12494,18,2025-06-02,2025-06-30
6,juli.csv,12515,23,2025-07-01,2025-07-31
7,agustus.csv,12518,20,2025-08-01,2025-08-29
8,september.csv,12548,21,2025-09-01,2025-09-30
9,oktober.csv,12549,23,2025-10-01,2025-10-31


Total baris sumber: 150,207


In [7]:
dataset_overview.to_csv(OUTPUT_DIR / 'dataset_overview.csv', index=False, encoding='utf-8-sig')
dataset_overview.describe()

,jumlah_baris,jumlah_tanggal
count,12.000000,12.000000
mean,12517.250000,19.750000
std,32.283193,2.137331
min,12473.000000,16.000000
25%,12490.250000,18.750000
50%,12518.000000,20.000000
75%,12548.250000,21.000000
max,12560.000000,23.000000


## 3. Transformasi dan pembersihan

Kolom identitas dirapikan dan diseragamkan, kode wilayah/komoditas tetap berupa teks agar formatnya terjaga, sedangkan harga diubah menjadi numerik. Harga kosong, bukan angka, nol, atau negatif serta baris tanpa identitas penting ditolak. Baris duplikat berdasarkan wilayah–komoditas–tanggal disisakan satu.

In [3]:
def normalize_code(values):
    return values.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)


def transform_month(path):
    raw = pd.read_csv(path, low_memory=False)
    date_columns = [column for column in raw.columns if column not in REQUIRED_COLUMNS]

    data = raw.rename(columns=COLUMNS_RENAME).drop(columns=['No']).copy()
    for column in ['kode_provinsi', 'kode_daerah', 'kode_komoditas']:
        data[column] = normalize_code(data[column])
    for column in ['provinsi', 'kabupaten_kota']:
        data[column] = data[column].astype('string').str.strip()
    data['komoditas'] = data['komoditas'].astype('string').str.strip().str.upper()

    long = data.melt(
        id_vars=list(COLUMNS_RENAME.values()),
        value_vars=date_columns,
        var_name='tanggal_sumber',
        value_name='harga',
    )
    long['tanggal'] = pd.to_datetime(long['tanggal_sumber'], format='%d/%m/%y', errors='coerce')
    long['harga'] = pd.to_numeric(long['harga'], errors='coerce')

    identifiers = list(COLUMNS_RENAME.values())
    has_identifiers = long[identifiers].notna().all(axis=1) & long[identifiers].ne('').all(axis=1)
    valid = has_identifiers & long['tanggal'].notna() & long['harga'].gt(0)
    quality = {
        'baris_kandidat': len(long),
        'harga_kosong_atau_non_numerik': int(long['harga'].isna().sum()),
        'harga_nol_atau_negatif': int((long['harga'].notna() & long['harga'].le(0)).sum()),
        'baris_tanpa_identitas': int((~has_identifiers).sum()),
    }

    cleaned = long.loc[valid, identifiers + ['tanggal', 'harga']].copy()
    duplicate_mask = cleaned.duplicated(['kode_daerah', 'kode_komoditas', 'tanggal'], keep='last')
    quality['duplikat_dihapus'] = int(duplicate_mask.sum())
    cleaned = cleaned.loc[~duplicate_mask].sort_values(
        ['tanggal', 'kode_daerah', 'kode_komoditas']
    ).reset_index(drop=True)
    quality['baris_diterima'] = len(cleaned)
    quality['baris_ditolak'] = len(long) - len(cleaned)
    return cleaned, quality


print('Fungsi transformasi siap.')

Fungsi transformasi siap.


## 4. Jalankan ETL awal dan simpan hasil lokal

Setiap bulan diproses lalu disimpan sebagai CSV untuk menghemat ruang penyimpanan. Ringkasan kualitas data disimpan di `etl_log.csv`. File sumber tidak diubah.

In [8]:
def run_etl():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summaries = []

    for path in INPUT_FILES:
        cleaned, quality = transform_month(path)
        output_path = OUTPUT_DIR / f'{path.stem}_cleaned.csv'
        cleaned.to_csv(output_path, index=False, date_format='%Y-%m-%d')
        summaries.append({
            'file_sumber': path.name,
            'file_hasil': output_path.name,
            **quality,
        })

    summary = pd.DataFrame(summaries)
    summary.to_csv(OUTPUT_DIR / 'etl_log.csv', index=False)
    return summary


etl_summary = run_etl()
display(etl_summary)
print(f"Total observasi hasil ETL: {etl_summary['baris_diterima'].sum():,}")
print(f"Hasil dan log tersimpan di: {OUTPUT_DIR}")

,file_sumber,file_hasil,baris_kandidat,harga_kosong_atau_non_numerik,harga_nol_atau_negatif,baris_tanpa_identitas,duplikat_dihapus,baris_diterima,baris_ditolak
0,januari.csv,januari_cleaned.csv,237101,317,0,19,0,236784,317
1,februari.csv,februari_cleaned.csv,250360,1637,0,20,0,248723,1637
2,maret.csv,maret_cleaned.csv,237899,1593,0,19,0,236306,1593
3,april.csv,april_cleaned.csv,199568,562,0,16,0,199006,562
4,mei.csv,mei_cleaned.csv,212041,490,0,17,0,211551,490
5,juni.csv,juni_cleaned.csv,224892,832,0,18,0,224060,832
6,juli.csv,juli_cleaned.csv,287845,726,0,23,0,287119,726
7,agustus.csv,agustus_cleaned.csv,250360,213,0,20,0,250147,213
8,september.csv,september_cleaned.csv,263508,339,0,21,0,263169,339
9,oktober.csv,oktober_cleaned.csv,288627,51,0,0,0,288576,51


Total observasi hasil ETL: 2,959,822
Hasil dan log tersimpan di: C:\TUGAS\Semester5\BANJUT\PanganPulse\etl\data\processed


In [10]:
from pathlib import Path
import pandas as pd

cleaned_files = sorted(OUTPUT_DIR.glob("*_cleaned.csv"))

prices_all = pd.concat(
    [pd.read_csv(file) for file in cleaned_files],
    ignore_index=True
)

print("Jumlah file:", len(cleaned_files))
print("Total observasi:", len(prices_all))

display(prices_all.head())

Jumlah file: 12
Total observasi: 2959822


,kode_provinsi,provinsi,kode_daerah,kabupaten_kota,kode_komoditas,komoditas,tanggal,harga
0,11,Aceh,1101,Kab. Simeulue,111001001,BERAS PREMIUM,2025-08-01,17600.0
1,11,Aceh,1101,Kab. Simeulue,111001002,BERAS MEDIUM,2025-08-01,17000.0
2,11,Aceh,1101,Kab. Simeulue,111008,TEPUNG TERIGU,2025-08-01,15000.0
3,11,Aceh,1101,Kab. Simeulue,111047,MIE KERING INSTANT,2025-08-01,3000.0
4,11,Aceh,1101,Kab. Simeulue,112003,DAGING AYAM RAS,2025-08-01,43000.0


In [11]:
# Struktur dataset
prices_all.info()

# Statistik harga
display(prices_all['harga'].describe())

# Jumlah wilayah dan komoditas
print("Jumlah wilayah:", prices_all['kode_daerah'].nunique())
print("Jumlah komoditas:", prices_all['komoditas'].nunique())

# Periksa missing values
display(prices_all.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2959822 entries, 0 to 2959821
Data columns (total 8 columns):
 #   Column          Dtype  
---  ------          -----  
 0   kode_provinsi   int64  
 1   provinsi        object 
 2   kode_daerah     int64  
 3   kabupaten_kota  object 
 4   kode_komoditas  int64  
 5   komoditas       object 
 6   tanggal         object 
 7   harga           float64
dtypes: float64(1), int64(3), object(4)
memory usage: 180.7+ MB


count    2.959822e+06
mean     3.012323e+04
std      2.904422e+04
min      1.000000e+03
25%      1.200000e+04
50%      1.833333e+04
75%      4.000000e+04
max      2.166667e+05
Name: harga, dtype: float64

Jumlah wilayah: 514
Jumlah komoditas: 27


kode_provinsi     0
provinsi          0
kode_daerah       0
kabupaten_kota    0
kode_komoditas    0
komoditas         0
tanggal           0
harga             0
dtype: int64